# Step 1
 - Extracting Commit Messages for our commit list from the fully cloned repos saved locally

In [3]:
from __future__ import annotations

import csv
import re
import subprocess
from pathlib import Path
from typing import Dict, Set, Optional, Tuple
from collections import defaultdict

# -----------------------------
# Config (edit as needed)
# -----------------------------
CLONE_ROOT = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Clone")

TARGET_COMMITS_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv"
)

# Write output alongside your existing metadata outputs (same folder as PR/Issue JSONs)
OUT_COMMIT_CSV = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\CommitMessages_local.csv"
)

# -----------------------------
# Helpers
# -----------------------------

GITHUB_RE = re.compile(r"github\.com[:/]+([^/]+)/([^/]+?)(?:\.git)?$", re.IGNORECASE)

def parse_owner_repo_from_remote(remote_url: str) -> Optional[Tuple[str, str]]:
    """
    Parse owner/repo from typical GitHub remote URLs:
      - https://github.com/owner/repo(.git)
      - git@github.com:owner/repo(.git)
    """
    u = (remote_url or "").strip()
    if not u:
        return None
    m = GITHUB_RE.search(u)
    if not m:
        return None
    return m.group(1), m.group(2)

def load_target_commits(path: Path) -> Dict[str, Set[str]]:
    """
    Load commits per repo from CSV columns:
      - repo_name: expected "owner__repo"
      - commit: sha
    """
    mapping: Dict[str, Set[str]] = defaultdict(set)
    if not path.exists():
        raise FileNotFoundError(f"Target commit CSV not found: {path}")

    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            repo_name = (row.get("repo_name") or "").strip()
            sha = (row.get("commit_sha") or "").strip()
            if repo_name and sha:
                mapping[repo_name].add(sha)

    print(f"Loaded target commits for {len(mapping)} repos from: {path}")
    return mapping

def run_git(repo_path: Path, args: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(
        ["git", "-C", str(repo_path), *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )


def is_git_repo(path: Path) -> bool:
    """
    Determine if path is a git repo.
    """
    if (path / ".git").exists():
        return True
    p = run_git(path, ["rev-parse", "--is-inside-work-tree"])
    return p.returncode == 0 and p.stdout.strip().lower() == "true"

def get_origin_url(repo_path: Path) -> str:
    """
    Get origin remote URL if available.
    """
    p = run_git(repo_path, ["remote", "get-url", "origin"])
    if p.returncode == 0:
        return p.stdout.strip()
    # fallback: try config key
    p2 = run_git(repo_path, ["config", "--get", "remote.origin.url"])
    return p2.stdout.strip() if p2.returncode == 0 else ""

def index_local_clones(clone_root: Path) -> Dict[str, Path]:
    """
    Build mapping: repo_key ("owner__repo") -> local path.
    Uses origin remote when possible; falls back to folder name if needed.
    """
    if not clone_root.exists():
        raise FileNotFoundError(f"Clone root not found: {clone_root}")

    mapping: Dict[str, Path] = {}

    # Assumption: clones are direct children; if yours are nested, change to rglob("*")
    for child in clone_root.iterdir():
        if not child.is_dir():
            continue
        if not is_git_repo(child):
            continue

        origin = get_origin_url(child)
        parsed = parse_owner_repo_from_remote(origin)
        if parsed:
            owner, repo = parsed
            key = f"{owner}__{repo}"
            mapping[key] = child
        else:
            # fallback to folder name
            mapping[child.name] = child

    print(f"Indexed {len(mapping)} local git repos under: {clone_root}")
    return mapping

def get_commit_message(repo_path: Path, sha: str) -> Tuple[Optional[dict], str]:
    """
    Return (commit_record_dict, error_message).
    commit_record_dict includes subject/body + author/committer metadata.
    """
    # Use US (unit separator) to split fields safely
    sep = "\x1f"
    fmt = sep.join([
        "%H",     # sha
        "%an",    # author name
        "%ae",    # author email
        "%ad",    # author date
        "%cn",    # committer name
        "%ce",    # committer email
        "%cd",    # committer date
        "%s",     # subject
        "%b",     # body (no subject)
    ])

    p = run_git(repo_path, ["show", "-s", f"--date=iso-strict", f"--format={fmt}", sha])
    if p.returncode != 0:
        err = (p.stderr or p.stdout or "").strip()
        return None, err[:2000]

    raw = p.stdout.rstrip("\n")
    parts = raw.split(sep)

    if len(parts) < 9:
        return None, f"Unexpected git output format for {sha}. Raw={raw[:200]}"

    (sha_out, an, ae, ad, cn, ce, cd, subject, body) = parts[:9]

    full_message = subject if not body.strip() else (subject + "\n\n" + body.rstrip())

    rec = {
        "commit_sha": sha_out,
        "author_name": an,
        "author_email": ae,
        "author_date": ad,
        "committer_name": cn,
        "committer_email": ce,
        "committer_date": cd,
        "subject": subject,
        "body": body,
        "full_message": full_message,
    }
    return rec, ""

# -----------------------------
# Main
# -----------------------------
def main() -> None:
    targets = load_target_commits(TARGET_COMMITS_CSV)
    repo_map = index_local_clones(CLONE_ROOT)

    OUT_COMMIT_CSV.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = [
        # PRIMARY KEYS for merge
        "repo_name",       # e.g., owner__repo
        "commit_sha",      # full sha
        # useful metadata
        "subject",
        "body",
        "full_message",
        "author_name",
        "author_email",
        "author_date",
        "committer_name",
        "committer_email",
        "committer_date",
        # traceability
        "local_repo_path",
        "status",
        "error",
    ]

    out_rows = []
    missing_repo = 0
    missing_commit = 0
    ok = 0

    for repo_name, shas in targets.items():
        repo_path = repo_map.get(repo_name)

        if repo_path is None:
            # could not find the repo locally
            for sha in shas:
                out_rows.append({
                    "repo_name": repo_name,
                    "commit_sha": sha,
                    "subject": "",
                    "body": "",
                    "full_message": "",
                    "author_name": "",
                    "author_email": "",
                    "author_date": "",
                    "committer_name": "",
                    "committer_email": "",
                    "committer_date": "",
                    "local_repo_path": "",
                    "status": "missing_repo",
                    "error": "Repo not found under CLONE_ROOT (by origin URL or folder name).",
                })
            missing_repo += 1
            continue

        for sha in sorted(shas):
            rec, err = get_commit_message(repo_path, sha)
            if rec is None:
                out_rows.append({
                    "repo_name": repo_name,
                    "commit_sha": sha,
                    "subject": "",
                    "body": "",
                    "full_message": "",
                    "author_name": "",
                    "author_email": "",
                    "author_date": "",
                    "committer_name": "",
                    "committer_email": "",
                    "committer_date": "",
                    "local_repo_path": str(repo_path),
                    "status": "missing_commit",
                    "error": err or "Commit not found in local clone (shallow clone or missing history).",
                })
                missing_commit += 1
            else:
                out_rows.append({
                    "repo_name": repo_name,
                    "commit_sha": rec["commit_sha"],
                    "subject": rec["subject"],
                    "body": rec["body"],
                    "full_message": rec["full_message"],
                    "author_name": rec["author_name"],
                    "author_email": rec["author_email"],
                    "author_date": rec["author_date"],
                    "committer_name": rec["committer_name"],
                    "committer_email": rec["committer_email"],
                    "committer_date": rec["committer_date"],
                    "local_repo_path": str(repo_path),
                    "status": "ok",
                    "error": "",
                })
                ok += 1

    with OUT_COMMIT_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(out_rows)

    print(f"Done. ok={ok}, missing_repo_groups={missing_repo}, missing_commit_rows={missing_commit}")
    print(f"Wrote: {OUT_COMMIT_CSV}")

if __name__ == "__main__":
    main()


Loaded target commits for 398 repos from: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv
Indexed 477 local git repos under: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Clone
Done. ok=494, missing_repo_groups=4, missing_commit_rows=3
Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\CommitMessages_local.csv


# Step 2 - Extracting PR messages as well as the issues

The first step in data process for this RQ starts with reading the commit and PR messages and issues from list of commits detected as instru related commits from the prevoius RQ

In [ ]:
from __future__ import annotations
import csv, os, re, time, json
from pathlib import Path
from typing import Optional, List, Dict, Set
from collections import defaultdict

import requests          # pip install requests
from dotenv import load_dotenv  # pip install python-dotenv

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT    = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2")
URL_LIST_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\URL_List.csv")
MANIFEST_CSV = WORK_ROOT / "PRs_Issues_manifest.csv"

# Where to store GitHub metadata (PRs + issues)
META_ROOT = WORK_ROOT / "PRs_Issues"

# NEW: path to the commit list produced by your episode analysis
TARGET_COMMITS_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv") 

FETCH_GH_METADATA = True  # set False to dry-run the URL list

# Path to your env file
ENV_FILE = r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env"

# === LOAD .env ===
load_dotenv(ENV_FILE)

# Load GitHub tokens: GITHUB_TOKEN_1 ... GITHUB_TOKEN_6
TOKENS = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
TOKENS = [t for t in TOKENS if t]

if not TOKENS:
    print("⚠️ Warning: No GitHub tokens found in All_Tokens.env (API metadata might be rate limited).")
else:
    print(f"ℹ️ Loaded {len(TOKENS)} GitHub token(s) from All_Tokens.env")
    print("Token lengths:", [len(t) for t in TOKENS])

# index of the current token (0-based)
token_index = 0

GITHUB_API_BASE = "https://api.github.com"

# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
META_ROOT.mkdir(parents=True, exist_ok=True)

# Regex for issue references like "#123"
ISSUE_REF_RE = re.compile(r"#(\d+)")

# -----------------------------
# Load target commits per repo
# -----------------------------
def load_target_commits(path: Path) -> Dict[str, Set[str]]:
    """
    Load target commit SHAs per repo from obs3_1_change_episodes_commits.csv.

    Uses the 'repo_name' column (e.g., 'connectbot__connectbot') and 'commit'.
    """
    mapping: Dict[str, Set[str]] = defaultdict(set)
    if not path.exists():
        print(f"⚠️ Warning: target commit CSV not found: {path}")
        return mapping

    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            repo_name = (row.get("repo_name") or "").strip()  # e.g., "connectbot__connectbot"
            sha = (row.get("commit_sha") or "").strip()
            if repo_name and sha:
                mapping[repo_name].add(sha)

    print(f"ℹ️ Loaded target commits for {len(mapping)} repos from {path}")
    return mapping


TARGET_COMMITS_BY_REPO: Dict[str, Set[str]] = load_target_commits(TARGET_COMMITS_CSV)

# -----------------------------
# Helpers (no git here)
# -----------------------------

def parse_github_owner_repo(url: str) -> Optional[tuple[str, str]]:
    """
    Return (owner, repo) for GitHub URLs, or None if not GitHub.
    Supports https://github.com/owner/repo(.git) and git@github.com:owner/repo(.git).
    """
    u = url.strip()
    # SSH form
    m = re.match(r"^git@github\.com:([^/]+)/(.+?)(?:\.git)?$", u)
    if m:
        return m.group(1), m.group(2)

    # HTTPS form
    if "github.com" not in u.lower():
        return None

    # Drop protocol
    if "://" in u:
        base = u.split("://", 1)[1]
    else:
        base = u

    # Remove possible query/fragments
    base = base.split("?", 1)[0].split("#", 1)[0]

    parts = [p for p in base.split("/") if p]
    # parts like ["github.com", "owner", "repo(.git)"]
    if len(parts) >= 3 and parts[0].lower().startswith("github.com"):
        owner = parts[1]
        repo = parts[2].removesuffix(".git")
        return owner, repo

    return None


def github_get(path: str, params: Optional[dict] = None) -> list:
    """
    Basic GitHub API GET with pagination.
    Uses round-robin token rotation when hitting rate limits.
    Returns a list of items.
    """
    global token_index

    url = f"{GITHUB_API_BASE}{path}"
    items: list = []
    page = 1

    while True:
        # Build headers with the current token
        headers = {
            # include preview for /commits/{sha}/pulls endpoint as well
            "Accept": "application/vnd.github+json, application/vnd.github.groot-preview+json"
        }
        current_token = TOKENS[token_index] if TOKENS else None
        if current_token:
            headers["Authorization"] = f"Bearer {current_token}"

        q = dict(params or {})
        q.setdefault("per_page", 100)
        q["page"] = page

        resp = requests.get(url, headers=headers, params=q)

        # Detect rate limit
        remaining = resp.headers.get("X-RateLimit-Remaining")
        is_rate_limited = (
            resp.status_code == 403
            and ("rate limit" in resp.text.lower() or remaining == "0")
        )

        if is_rate_limited:
            print(
                f"[rate limit] {path} page={page} with token index {token_index}. "
                f"Remaining={remaining}"
            )
            if TOKENS and len(TOKENS) > 1:
                old_index = token_index
                token_index = (token_index + 1) % len(TOKENS)
                print(f"  -> switching token {old_index} -> {token_index} and retrying...")
                continue
            else:
                print("  -> no alternative tokens; stopping.")
                break

        # If we reach here, it's not a rate-limit error
        resp.raise_for_status()
        data = resp.json()
        if not data:
            break

        if isinstance(data, list):
            items.extend(data)
        else:
            # some endpoints return an object, not a list
            items.append(data)
            break

        if len(data) < q["per_page"]:
            # last page
            break

        page += 1

    return items


# --- Targeted helpers: commit -> PRs, PR -> issues ----

def get_prs_for_commit(owner: str, repo: str, sha: str) -> List[dict]:
    """
    Return list of PRs that include this commit.
    Uses /repos/{owner}/{repo}/commits/{sha}/pulls.
    """
    path = f"/repos/{owner}/{repo}/commits/{sha}/pulls"
    try:
        return github_get(path, params=None)
    except Exception as e:
        print(f"    [warn] commit {sha}: error fetching associated PRs: {e}")
        return []


def fetch_issue_with_comments(owner: str, repo: str, number: int) -> Optional[dict]:
    """
    Fetch a single issue + its comments.
    """
    try:
        issue_list = github_get(f"/repos/{owner}/{repo}/issues/{number}")
        if not issue_list:
            return None
        issue = issue_list[0]
    except Exception as e:
        print(f"    [warn] issue #{number}: error fetching issue: {e}")
        return None

    try:
        comments = github_get(f"/repos/{owner}/{repo}/issues/{number}/comments")
    except Exception as e:
        print(f"    [warn] issue #{number}: error fetching comments: {e}")
        comments = []
    issue["comments"] = comments
    return issue


def extract_issue_numbers_from_pr(pr: dict) -> Set[int]:
    """
    Extract referenced issue numbers from PR body/title/comments/reviews using #123 pattern.
    """
    texts: List[str] = []

    for key in ("title", "body"):
        v = pr.get(key)
        if isinstance(v, str):
            texts.append(v)

    for c in pr.get("issue_comments", []):
        v = c.get("body")
        if isinstance(v, str):
            texts.append(v)

    for c in pr.get("review_comments", []):
        v = c.get("body")
        if isinstance(v, str):
            texts.append(v)

    for r in pr.get("reviews", []):
        v = r.get("body")
        if isinstance(v, str):
            texts.append(v)

    nums: Set[int] = set()
    for t in texts:
        for m in ISSUE_REF_RE.findall(t):
            try:
                nums.add(int(m))
            except ValueError:
                pass
    return nums


def fetch_github_prs_and_issues(owner: str, repo: str, out_dir: Path) -> tuple[int, int]:
    """
    TARGETED VERSION:

    For this (owner, repo), we only fetch metadata for the commits listed
    in obs3_1_change_episodes_commits.csv (per repo).

    For those commits:
      - find associated PRs
      - fetch full PR (details + issue_comments + review_comments + reviews)
      - parse PR text/comments to find #issue references
      - fetch only those issues (+ comments)

    Returns:
        (num_targeted_prs, num_targeted_issues)
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    base_name = f"{owner}__{repo}"

    repo_key = base_name  # matches 'repo_name' in the commits CSV
    commit_shas = TARGET_COMMITS_BY_REPO.get(repo_key, set())

    if not commit_shas:
        print(f"  [meta] No target commits for {owner}/{repo} (repo_name={repo_key}); skipping.")
        # still write empty JSON files for consistency
        with (out_dir / f"{base_name}_PRs.json").open("w", encoding="utf-8") as f:
            json.dump([], f, ensure_ascii=False, indent=2)
        with (out_dir / f"{base_name}_Issues.json").open("w", encoding="utf-8") as f:
            json.dump([], f, ensure_ascii=False, indent=2)
        return 0, 0

    print(f"  [meta] Target commits for {owner}/{repo}: {len(commit_shas)}")

    targeted_prs: List[dict] = []
    targeted_issues: List[dict] = []

    seen_pr_numbers: Set[int] = set()
    seen_issue_numbers: Set[int] = set()

    # ---- 1) For each commit, get its PRs and fetch full PR discussion ----
    for sha in sorted(commit_shas):
        prs_for_commit = get_prs_for_commit(owner, repo, sha)
        if not prs_for_commit:
            continue

        for pr_stub in prs_for_commit:
            number = pr_stub.get("number")
            if number is None:
                continue
            if number in seen_pr_numbers:
                # Optionally attach this commit to an existing PR record
                for existing in targeted_prs:
                    if existing.get("number") == number:
                        existing.setdefault("target_commits", [])
                        if sha not in existing["target_commits"]:
                            existing["target_commits"].append(sha)
                continue

            seen_pr_numbers.add(number)

            # PR core details
            try:
                pr_full_list = github_get(f"/repos/{owner}/{repo}/pulls/{number}")
                pr_full = pr_full_list[0] if pr_full_list else pr_stub
            except Exception as e:
                print(f"    [warn] PR #{number}: error fetching details: {e}")
                pr_full = pr_stub

            # issue-style comments on the PR
            try:
                issue_comments = github_get(
                    f"/repos/{owner}/{repo}/issues/{number}/comments"
                )
            except Exception as e:
                print(f"    [warn] PR #{number}: error fetching issue comments: {e}")
                issue_comments = []

            # review comments on specific lines
            try:
                review_comments = github_get(
                    f"/repos/{owner}/{repo}/pulls/{number}/comments"
                )
            except Exception as e:
                print(f"    [warn] PR #{number}: error fetching review comments: {e}")
                review_comments = []

            # review events (approve/request-changes etc.)
            try:
                reviews = github_get(
                    f"/repos/{owner}/{repo}/pulls/{number}/reviews"
                )
            except Exception as e:
                print(f"    [warn] PR #{number}: error fetching reviews: {e}")
                reviews = []

            pr_full["issue_comments"] = issue_comments
            pr_full["review_comments"] = review_comments
            pr_full["reviews"] = reviews
            pr_full["target_commits"] = [sha]

            targeted_prs.append(pr_full)

    # ---- 2) From those PRs, find referenced issues and fetch them ----
    for pr in targeted_prs:
        issue_numbers = extract_issue_numbers_from_pr(pr)
        for num in issue_numbers:
            if num in seen_issue_numbers:
                continue
            seen_issue_numbers.add(num)
            issue = fetch_issue_with_comments(owner, repo, num)
            if issue is not None:
                targeted_issues.append(issue)

    # ---- 3) Save as JSON (same filenames as before) ----
    with (out_dir / f"{base_name}_PRs.json").open("w", encoding="utf-8") as f:
        json.dump(targeted_prs, f, ensure_ascii=False, indent=2)

    with (out_dir / f"{base_name}_Issues.json").open("w", encoding="utf-8") as f:
        json.dump(targeted_issues, f, ensure_ascii=False, indent=2)

    return len(targeted_prs), len(targeted_issues)


# -----------------------------
# Main: GH METADATA ONLY (no clone)
# -----------------------------
def main() -> None:
    assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

    rows, ok, fail = [], 0, 0
    with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            url = (row.get("repo_url") or "").strip()
            if not url:
                continue
            t0 = time.time()
            rec = {
                "repo_url": url,
                "status": "unknown",
                "seconds": None,
                "num_prs": None,
                "num_issues": None,
                "error": "",
            }
            try:
                if not FETCH_GH_METADATA:
                    rec["status"] = "skipped"
                    ok += 1
                else:
                    gh = parse_github_owner_repo(url)
                    if gh is None:
                        rec["status"] = "non_github"
                        rec["error"] = "Not a GitHub URL"
                        fail += 1
                    else:
                        owner, repo_name = gh
                        try:
                            num_prs, num_issues = fetch_github_prs_and_issues(
                                owner, repo_name, META_ROOT
                            )
                            rec["status"] = "ok"
                            rec["num_prs"] = num_prs
                            rec["num_issues"] = num_issues
                            ok += 1
                        except Exception as e:
                            msg = f"metadata error: {e}"
                            print(f"  [meta-error] {url}: {msg}")
                            rec["status"] = "error"
                            rec["error"] = msg
                            fail += 1

            except Exception as e:
                rec["status"] = "error"
                rec["error"] = str(e)[:2000]
                fail += 1

            rec["seconds"] = round(time.time() - t0, 2)
            rows.append(rec)
            print(
                f"[{rec['status']}] {url} "
                f"({rec['seconds']}s)  PRs={rec['num_prs']}  Issues={rec['num_issues']}"
            )

    # Write manifest for metadata
    MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["repo_url", "status", "seconds", "num_prs", "num_issues", "error"]
    with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


if __name__ == "__main__":
    main()


ℹ️ Loaded 6 GitHub token(s) from All_Tokens.env
Token lengths: [40, 40, 40, 40, 40, 40]
ℹ️ Loaded target commits for 398 repos from C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv
  [meta] Target commits for connectbot/connectbot: 2
[ok] https://github.com/connectbot/connectbot (2.97s)  PRs=2  Issues=0
  [meta] Target commits for robolectric/robolectric: 2
[ok] https://github.com/robolectric/robolectric (4.91s)  PRs=2  Issues=0
  [meta] Target commits for opendocument-app/OpenDocument.droid: 1
[ok] https://github.com/opendocument-app/OpenDocument.droid (1.44s)  PRs=1  Issues=0
  [meta] Target commits for maxpower47/PinDroid: 1
[ok] https://github.com/maxpower47/PinDroid (1.56s)  PRs=1  Issues=0
  [meta] Target commits for Rajawali/Rajawali: 3
[ok] https://github.com/Rajawali/Rajawali (6.33s)  PRs=3  Issues=3
  [meta] Target commits for cgeo/cgeo: 1
[ok] https://github.com/cgeo/cgeo (1.82s)  PRs=1  Issues=0
 

# Step 5: Final CSV: 

Final list of change commits with all the intention sources: Commits subjects and boddys, PRs and Issues

## Step 5 - appending the commit and PR messages to the episode lists

After having change episods list "list_of_episodes" and "All_Env_Change_Commit_PR_Iss_Final". we merge them together in this code

## Step 4: Intention labeling by keywords

In [17]:
import pandas as pd

# -----------------------------
# Config: paths & column names
# -----------------------------
INPUT_PATH = "C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4\Observations\All_episodes_with_messages.csv"
OUTPUT_PATH = "C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4\Observations\All_episodes_with_messages_with_intentions_keywords.csv"

# Columns used to build the "start" and "end" text blobs
START_COLS = [
    "start_commit_subject",
    "start_commit_body",
    "start_pr_title",
    "start_pr_body",
    "start_issue_summary",
]

END_COLS = [
    "end_commit_subject",
    "end_commit_body",
    "end_pr_title",
    "end_pr_body",
    "end_issue_summary",
]

END_COMMIT_COL = "episode_end_commit_sha"  # used to decide if end_intention should be null


# -----------------------------
# Keyword dictionaries
# -----------------------------
ci_keywords = [
    "ci", "github actions", "action", "actions", "workflow", "workflows",
    "travis", "circleci", "bitrise", "jenkins", "gitlab ci", "buildkite",
    "azure pipelines", "pipeline", "runner", "job", "jobs",
]

test_keywords = [
    "test", "tests", "testing", "unittest", "unit test", "unit tests",
    "androidtest", "connectedandroidtest", "instrumentation", "instrumentation test",
    "instrumentation tests", "ui test", "ui tests", "uitest", "espresso",
    "robolectric", "detox", "snapshot test", "screenshot test", "coverage",
    "jacoco", "lint test", "integration test", "integration tests",
    "e2e", "end-to-end", "end to end", "functional test", "acceptance test",
]

migrate_keywords = [
    "migrate", "migration", "move to", "moving to", "switch to", "switch from",
    "replace", "replaced", "rewrite", "rewrote", "port", "ported",
    "deprecate", "deprecated", "retire", "retired", "drop travis",
    "drop circleci", "drop bitrise", "new ci", "ci v2",
]

scope_keywords = [
    "e2e", "end-to-end", "end to end",
    "integration test", "integration tests",
    "instrumentation test", "instrumentation tests",
    "ui test", "ui tests", "uitest",
    "espresso", "detox", "firebase test lab", "ftl",
    "device farm", "device lab", "browserstack", "saucelabs", "sauce labs",
    "matrix", "multi-device", "multi device", "multiple devices",
    "gradle managed device", "managed device", "gmd", "benchmark",
    "screenshot test", "snapshot test", "golden test",
]

release_keywords = [
    "release", "releasing", "deploy", "deployment", "deploying",
    "publish", "publishing", "uploaded to", "upload to",
    "play store", "google play", "app store", "testflight",
    "beta", "production", "release pipeline", "release workflow",
    "version bump", "bump version", "bump to", "tag", "tagged",
    "artifact", "maven central", "jitpack", "fastlane",
    "signed apk", "bundle", "aab", "apk", "rollout", "roll out",
]

cleanup_keywords = [
    "clean up", "cleanup", "tidy", "tidied", "remove", "removed", "drop", "dropped",
    "delete", "deleted", "simplify", "simplified", "refactor", "refactored",
    "restructure", "restructured", "re-organize", "reorganize", "consolidate",
    "deduplicate", "dedup", "dedupe", "unused", "legacy", "obsolete",
    "dead code", "strip", "trim", "split workflow", "combine workflow",
    "rename workflow", "rename job", "cleanup ci", "clean ci",
]

env_keywords = [
    "emulator", "emulators", "emu", "device", "devices", "avd",
    "virtual device", "gmd", "managed device", "test matrix",
]

perf_keywords = [
    "flaky", "flakiness", "flake", "deflake", "unstable", "stability",
    "stable", "more stable", "less stable", "less flaky",
    "fixed flaky", "slow", "slowness", "slowdown", "faster",
    "speed up", "speedup", "performance", "perf", "timeout",
    "time-out", "time out", "hang", "hanging", "hangs",
    "crash", "crashes", "crashing", "intermittent", "intermittently",
    "non-deterministic", "non deterministic", "sporadic",
    "retry", "retries", "rerun", "re-run", "stuck", "stalls", "stalled",
]


# -----------------------------
# Detection helpers
# -----------------------------
def has_any(text: str, keywords) -> bool:
    """Return True if any keyword is a substring of text."""
    return any(kw in text for kw in keywords)


def detect_intentions_from_text(text: str):
    """
    Return a comma-separated string of intention labels
    for the given text, or None if nothing matches.
    """
    if not isinstance(text, str):
        return None
    t = text.lower().strip()
    if not t:
        return None

    labels = set()

    # 1. Introduce / strengthen CI-backed tests
    if has_any(t, ci_keywords) and has_any(t, test_keywords):
        labels.add("Introduce / strengthen CI-backed tests")

    # 2. Migrate or modernise CI infrastructure
    if has_any(t, migrate_keywords) and has_any(t, ci_keywords):
        labels.add("Migrate or modernise CI infrastructure")

    # 3. Expand test scope or capabilities
    if has_any(t, scope_keywords):
        labels.add("Expand test scope or capabilities")

    # 4. Automate or integrate release workflows
    if has_any(t, release_keywords):
        labels.add("Automate or integrate release workflows")

    # 5. Clean up or simplify CI / environment configuration
    if has_any(t, cleanup_keywords) and (has_any(t, ci_keywords) or has_any(t, env_keywords)):
        labels.add("Clean up or simplify CI / environment configuration")

    # 6. Address performance or stability issues
    if has_any(t, perf_keywords):
        labels.add("Address performance or stability issues")

    if not labels:
        return None

    # Sorted for determinism; comma-separated for multiple labels
    return ", ".join(sorted(labels))


def build_text(row, cols):
    """Concatenate non-empty text from the given columns for a row."""
    parts = []
    for c in cols:
        if c in row:
            val = row[c]
            if isinstance(val, str) and val.strip():
                parts.append(val.strip())
    return "\n".join(parts)


# -----------------------------
# Main processing
# -----------------------------
def main():
    df = pd.read_csv(INPUT_PATH)

    # start_intention: always computed from available start-* fields
    df["start_intention"] = df.apply(
        lambda r: detect_intentions_from_text(build_text(r, START_COLS)),
        axis=1,
    )

    # end_intention: only if there *is* an end commit; otherwise keep as None/NaN
    def compute_end_intention(row):
        if pd.isna(row.get(END_COMMIT_COL)):
            return None
        end_text = build_text(row, END_COLS)
        return detect_intentions_from_text(end_text)

    df["end_intention"] = df.apply(compute_end_intention, axis=1)

    # Save the updated file
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved updated file with intentions to: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


Saved updated file with intentions to: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ4\Observations\All_episodes_with_messages_with_intentions_keywords.csv
